In [0]:
df = spark.read.csv("/Volumes/workspace/dbo/bronze_lawyer/", header=True, inferSchema=True)
           #header=True  it will remove header rows, inferSchema=True it will detect data types automatically
display(df)

order_id,order_date,customer_name,product,category,quantity,price
1001,2026-09-08,Emily,Blender,Home,1,55
1002,2026-09-09,Kavya,Keyboard,Electronics,2,55
1003,2026-10-08,Priya,Mouse,Electronics,1,20
1004,2026-09-15,Kavya,Keyboard,Electronics,1,40
1005,2026-10-16,Sneha,Keyboard,Electronics,2,50
1006,2026-09-29,Robert,Keyboard,Electronics,2,40
1007,2026-09-28,Suresh,Smartphone,Electronics,1,650
1008,2026-09-22,David,Jacket,Fashion,1,85
1009,2026-10-25,John,Headphones,Electronics,1,75
1010,2026-09-30,Priya,Smartphone,Electronics,1,650


In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [0]:
print("Number of records:", df.count()) #rows count

Number of records: 500


write to bronze table

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/dbo/bronze_lawyer/delta_orders/")

Final Record count

In [0]:
bronze = spark.read.option("header", "true").csv("/Volumes/workspace/dbo/bronze_lawyer/pyspark_orders_dataset.csv")
display(bronze)

order_id,order_date,customer_name,product,category,quantity,price
1001,2026-09-08,Emily,Blender,Home,1,55
1002,2026-09-09,Kavya,Keyboard,Electronics,2,55
1003,2026-10-08,Priya,Mouse,Electronics,1,20
1004,2026-09-15,Kavya,Keyboard,Electronics,1,40
1005,2026-10-16,Sneha,Keyboard,Electronics,2,50
1006,2026-09-29,Robert,Keyboard,Electronics,2,40
1007,2026-09-28,Suresh,Smartphone,Electronics,1,650
1008,2026-09-22,David,Jacket,Fashion,1,85
1009,2026-10-25,John,Headphones,Electronics,1,75
1010,2026-09-30,Priya,Smartphone,Electronics,1,650


In [0]:
print("Bronze record count", bronze.count())  #intha count find pannatha mela table read panni bronze la save panrom

Bronze record count 500


OrderID miss aaki irukannnu check panrom

In [0]:
import pyspark.sql.functions as f

bronze.filter(F.col("order_id").isNull()).show()

+--------+----------+-------------+-------+--------+--------+-----+
|order_id|order_date|customer_name|product|category|quantity|price|
+--------+----------+-------------+-------+--------+--------+-----+
+--------+----------+-------------+-------+--------+--------+-----+



change datatype in quantity and price

In [0]:
from pyspark.sql.functions import *

bronze = bronze.withColumn(
    "quantity", col("quantity").cast("int")
).withColumn(
    "price", col("price").cast("int")
)

Quantity la null value irukannu check panrom

In [0]:
import pyspark.sql.functions as f
bronze.filter(f.col("quantity").isNull()).show()


+--------+----------+-------------+-------+--------+--------+-----+
|order_id|order_date|customer_name|product|category|quantity|price|
+--------+----------+-------------+-------+--------+--------+-----+
+--------+----------+-------------+-------+--------+--------+-----+



check 0 or negative quantity

In [0]:

bronze.filter(f.col("quantity") <= 0).show()

+--------+----------+-------------+-------+--------+--------+-----+
|order_id|order_date|customer_name|product|category|quantity|price|
+--------+----------+-------------+-------+--------+--------+-----+
+--------+----------+-------------+-------+--------+--------+-----+



clean data

In [0]:
silver = bronze.filter(col("order_id").isNotNull())

silver = silver.filter(col("quantity") > 0 )

display(silver)

order_id,order_date,customer_name,product,category,quantity,price
1001,2026-09-08,Emily,Blender,Home,1,55
1002,2026-09-09,Kavya,Keyboard,Electronics,2,55
1003,2026-10-08,Priya,Mouse,Electronics,1,20
1004,2026-09-15,Kavya,Keyboard,Electronics,1,40
1005,2026-10-16,Sneha,Keyboard,Electronics,2,50
1006,2026-09-29,Robert,Keyboard,Electronics,2,40
1007,2026-09-28,Suresh,Smartphone,Electronics,1,650
1008,2026-09-22,David,Jacket,Fashion,1,85
1009,2026-10-25,John,Headphones,Electronics,1,75
1010,2026-09-30,Priya,Smartphone,Electronics,1,650


calculate Revenue

In [0]:
silver = silver.withColumn("Revenue",col("quantity") * col("price"))

display(silver)

order_id,order_date,customer_name,product,category,quantity,price,Revenue
1001,2026-09-08,Emily,Blender,Home,1,55,55
1002,2026-09-09,Kavya,Keyboard,Electronics,2,55,110
1003,2026-10-08,Priya,Mouse,Electronics,1,20,20
1004,2026-09-15,Kavya,Keyboard,Electronics,1,40,40
1005,2026-10-16,Sneha,Keyboard,Electronics,2,50,100
1006,2026-09-29,Robert,Keyboard,Electronics,2,40,80
1007,2026-09-28,Suresh,Smartphone,Electronics,1,650,650
1008,2026-09-22,David,Jacket,Fashion,1,85,85
1009,2026-10-25,John,Headphones,Electronics,1,75,75
1010,2026-09-30,Priya,Smartphone,Electronics,1,650,650


Compare the bronze and silver count

In [0]:
print("bronze recound count",bronze.count())
print("bronze recound count",silver.count())

bronze recound count 500
bronze recound count 500


save silver as a Delta table

In [0]:
silver.write.format("delta").mode("overwrite").save("/Volumes/workspace/dbo/silver_lawyer/")

check silver delta table saved or not

In [0]:
silver_table = spark.read.format("delta").load(
    "/Volumes/workspace/dbo/silver_lawyer/"
)

display(silver_table)

order_id,order_date,customer_name,product,category,quantity,price,Revenue
1001,2026-09-08,Emily,Blender,Home,1,55,55
1002,2026-09-09,Kavya,Keyboard,Electronics,2,55,110
1003,2026-10-08,Priya,Mouse,Electronics,1,20,20
1004,2026-09-15,Kavya,Keyboard,Electronics,1,40,40
1005,2026-10-16,Sneha,Keyboard,Electronics,2,50,100
1006,2026-09-29,Robert,Keyboard,Electronics,2,40,80
1007,2026-09-28,Suresh,Smartphone,Electronics,1,650,650
1008,2026-09-22,David,Jacket,Fashion,1,85,85
1009,2026-10-25,John,Headphones,Electronics,1,75,75
1010,2026-09-30,Priya,Smartphone,Electronics,1,650,650


In [0]:
print(silver.count())

500
